<a href="https://colab.research.google.com/github/nimraa9090/AI-projects/blob/main/FAKENEWS_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import re

In [7]:
# Load the datasets
df_fake = pd.read_csv(os.path.join(path, 'Fake.csv'))
df_true = pd.read_csv(os.path.join(path, 'True.csv'))

# Add class labels: 0 for Fake, 1 for True
df_fake['class'] = 0
df_true['class'] = 1

# Merge dataframes and drop unnecessary columns
df_merge = pd.concat([df_fake, df_true], axis=0)
df = df_merge.drop(['title', 'subject', 'date'], axis=1)
df = df.sample(frac=1) # Shuffle

# Text cleaning function
def wordopt(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['text'] = df['text'].apply(wordopt)

In [6]:
import kagglehub
import os

# Download latest version of the dataset
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

# The downloaded 'path' directly points to the directory containing the files.
# We can use this path to access the CSV files.
print(f"Dataset downloaded to: {path}")

# Print the path to the extracted files for verification
print("Contents of extracted directory:")
print(os.listdir(path))

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Dataset downloaded to: /kaggle/input/fake-and-real-news-dataset
Contents of extracted directory:
['True.csv', 'Fake.csv']


<!-- This cell previously contained instructions for re-running data loading, but this is no longer necessary as the data is now automatically downloaded and loaded. -->

In [10]:
X = df['text']
Y = df['class']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.25)

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(X_train)
xv_test = vectorization.transform(X_test)

In [11]:
LR = LogisticRegression()
LR.fit(xv_train, Y_train)

# Evaluate
pred_lr = LR.predict(xv_test)
print(classification_report(Y_test, pred_lr))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      5835
           1       0.98      0.99      0.99      5390

    accuracy                           0.99     11225
   macro avg       0.99      0.99      0.99     11225
weighted avg       0.99      0.99      0.99     11225



In [13]:
def manual_testing(news):
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(wordopt)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    pred = LR.predict(new_xv_test)

    return "Fake News" if pred[0] == 0 else "Not a Fake News"

# Example usage:
print(manual_testing("Donald Trump has announced he is running for president in 2024."))

Fake News
